# Event 4245 Live Chunk-by-Chunk Demo

Presentation-only notebook for the internal FABRIC/LoomAI Jupyter environment. It replays event 4245, watches Queue v2 chunk outputs update live, then shows final validation.

WIB = Warm Interface Board.


## Before You Run This Notebook

This notebook is an internal FABRIC/LoomAI presentation demo, not a full setup notebook.

It assumes all of the following are already true:

- The FABRIC slice already exists.
- The sender and receiver VMs are reachable from the internal FABRIC/LoomAI Jupyter environment.
- Required input files are already staged on the VMs.
- `configs/event_4245_replay.yaml` is already correct for the active slice, VM names, interfaces, PCAP path, mapping CSV, and validation reference CSV.
- VM dependencies are already installed.

This notebook does not create slices, submit slices, install dependencies, discover SSH hosts, or configure external SSH access.

If you need setup instructions, start with [notebooks/00_RUN_DEMO.ipynb](00_RUN_DEMO.ipynb) and [README.md](../README.md). External/local Jupyter remains dry-run verified only until tested with real SSH details.


## What Each Cell Does

| Section | Purpose |
| --- | --- |
| Configure Run | Loads `configs/event_4245_replay.yaml`, verifies the expected event 4245 baseline, builds the run ID, output paths, and backend command. |
| Optional Tick Playback Helper | Defines a presentation helper for separate tick-position playback; it is not part of live chunk progress. |
| Start Live Viewer | Starts the notebook display watcher before backend replay begins. |
| Start Backend | Starts the bounded internal LoomAI replay backend in the background. |
| Live Status + Final Validation | Waits for completion, reads manifest/summary files, and prints pass/fail details. |
| Per-Chunk Timing / Lag Table | Loads `reports/live_visualization_timing.csv` and summarizes capture/decode/copy/render timing. |
| Final Heatmap + Ridgeline | Displays the final live-viewer heatmap and ridgeline images. |

Run cells in order for the presentation path. Do not use this notebook as the first-run setup guide.


## 1. Configure Run

Defaults: `wib-fabric-live-demo`, event 4245, pcap replay, fixed Queue v2, 640 packets, 64 packets/chunk, 10 chunks. Large outputs stay under `/tmp/wib_fabric_live_demo/runs/<RUN_ID>/`.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import importlib.util
import inspect
import json
import shlex
import subprocess
import sys
import time

import pandas as pd
import yaml
from IPython.display import Image, display

NOTEBOOK_DIR = Path.cwd().resolve()
candidates = [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]
PACKAGE_ROOT = next(
    (
        candidate
        for candidate in candidates
        if (candidate / "scripts/run_pipeline.py").is_file()
        and (candidate / "configs/event_4245_replay.yaml").is_file()
    ),
    None,
)
if PACKAGE_ROOT is None:
    raise RuntimeError("Could not find the WIB/FABRIC package root. Start Jupyter from the artifact directory.")
CONFIG_PATH = PACKAGE_ROOT / "configs/event_4245_replay.yaml"

config = yaml.safe_load(CONFIG_PATH.read_text())
if config.get("slice", {}).get("name") == "obsolete-wib-demo-slice":
    raise RuntimeError("Do not use dead slice obsolete-wib-demo-slice; use wib-fabric-live-demo.")
if config.get("slice", {}).get("name") != "wib-fabric-live-demo":
    raise RuntimeError(f"Unexpected slice: {config.get('slice', {}).get('name')}")
if str(config.get("event_id")) != "4245":
    raise RuntimeError(f"Unexpected event: {config.get('event_id')}")
if config.get("source_mode") != "pcap_replay":
    raise RuntimeError(f"Unexpected source mode: {config.get('source_mode')}")
if config.get("decode", {}).get("queue", {}).get("implementation") != "queue_v2":
    raise RuntimeError("Expected fixed Queue v2 decode path.")
if config.get("decode", {}).get("adaptive", {}).get("enabled"):
    raise RuntimeError("Adaptive mode should be disabled for the default presentation demo.")

RUN_ID = f"presentation-live-4245-{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"
OUTPUT_ROOT = Path(config.get("paths", {}).get("output_root", "/tmp/wib_fabric_live_demo")).expanduser()
RUN_DIR = OUTPUT_ROOT / "runs" / RUN_ID
TOTAL_PACKETS = int(config["pcap_replay"]["packet_limit"])
CHUNK_PACKETS = int(config["decode"]["chunking"]["chunk_packets"])
TOTAL_CHUNKS = TOTAL_PACKETS // CHUNK_PACKETS

# Event 4245 full event extents from validated chunk outputs.
TICK_RANGE = (0, 2047)
FULL_EVENT_WIRE_RANGE = (0, 199)
WIRE_RANGE = FULL_EVENT_WIRE_RANGE
COLOR_LIMITS = (-20, 50)
WAVEFORM_MODE = "full_range"
WAVEFORM_WIRE_RANGE = FULL_EVENT_WIRE_RANGE
WAVEFORM_SAMPLE_COUNT = None
WAVEFORM_ADC_SCALE = 0.01  # matches benchmark signal_gain

backend_cmd = [
    sys.executable,
    str(PACKAGE_ROOT / "scripts/run_pipeline.py"),
    "--config",
    str(CONFIG_PATH),
    "--run-id",
    RUN_ID,
]

print("Run ID:", RUN_ID)
print("Run directory:", RUN_DIR)
print("Config:", CONFIG_PATH)
print(f"Event 4245: {TOTAL_PACKETS} packets / {CHUNK_PACKETS} packets per chunk = {TOTAL_CHUNKS} chunks")
print("Backend command:", " ".join(shlex.quote(str(part)) for part in backend_cmd))


## Optional Tick Playback Helper

Decoded event playback by tick position, not live chunk order. This is separate from live packet chunk progress.


In [ ]:
def visual_playback_by_tick(source_image_path, label="Decoded event playback by tick position, not live chunk order"):
    """Placeholder hook for presentation-only tick-position playback.

    Live chunk mode intentionally renders cumulative packet batches in actual decoded
    chunk order. Use this helper name only for a separate presentation animation that
    reveals an already-decoded event by tick position; do not use it for live progress.
    """
    print(label)
    print("Live chunk visualizations do not mask left-to-right by tick.")
    return source_image_path


## 2. Start Live Viewer

Run this cell before starting the backend. It watches decoded chunk CSVs and updates cumulative heatmap plus ridgeline in place. The timing CSV is written to `reports/live_visualization_timing.csv`.

Ordering/orientation note: the viewer updates by actual completed chunk order and does not artificially reveal the plot left-to-right. The event 4245 display range is fixed to wire_id 0-199, with wire_id 0 at the bottom and wire_id 199 at the top to match the original/GIF orientation. If an earlier notebook preview looked different, that was a plotting orientation/range issue, not a chunk-order failure.

For the presentation demo, both panels show the full event wire range, matching the benchmark/original visualization style. The ridgeline uses the benchmark scale: baseline at wire_id plus 0.01 * pedestal-subtracted ADC deviation.

Chunks are packet batches, not tick/time slices.

Raw ADC includes a large electronics baseline; the live demo subtracts a per-wire median pedestal so activity is shown as ADC deviation around zero.


In [ ]:
viewer_module_path = (PACKAGE_ROOT / "scripts/live_chunk_viewer.py").resolve()
spec = importlib.util.spec_from_file_location("live_chunk_viewer", viewer_module_path)
live_chunk_viewer = importlib.util.module_from_spec(spec)
sys.modules["live_chunk_viewer"] = live_chunk_viewer
spec.loader.exec_module(live_chunk_viewer)
start_notebook_live_viewer = live_chunk_viewer.start_notebook_live_viewer

print("live_chunk_viewer:", live_chunk_viewer.__file__)
print("signature:", inspect.signature(start_notebook_live_viewer))

live_viewer = start_notebook_live_viewer(
    RUN_DIR,
    poll_seconds=2.0,
    label=f"event 4245 {RUN_ID}",
    stop_when_complete=True,
    expected_total_chunks=TOTAL_CHUNKS,
    chunk_packets=CHUNK_PACKETS,
    total_packets=TOTAL_PACKETS,
    tick_range=TICK_RANGE,
    wire_range=WIRE_RANGE,
    color_limits=COLOR_LIMITS,
    waveform_mode=WAVEFORM_MODE,
    waveform_wire_range=WAVEFORM_WIRE_RANGE,
    waveform_window_size=WAVEFORM_SAMPLE_COUNT,
    waveform_adc_scale=WAVEFORM_ADC_SCALE,
)
print("Live viewer started. Now run the backend cell.")


## 3. Start Backend

This starts the bounded full event 4245 run in the background. The live viewer above should update as each decoded chunk CSV appears.


In [ ]:
report_dir = RUN_DIR / "reports"
report_dir.mkdir(parents=True, exist_ok=True)
backend_stdout_path = report_dir / "notebook_backend_stdout.log"
backend_stderr_path = report_dir / "notebook_backend_stderr.log"
backend_stdout_handle = backend_stdout_path.open("w", encoding="utf-8", buffering=1)
backend_stderr_handle = backend_stderr_path.open("w", encoding="utf-8", buffering=1)
backend_process = subprocess.Popen(
    backend_cmd,
    cwd=str(PACKAGE_ROOT),
    text=True,
    stdout=backend_stdout_handle,
    stderr=backend_stderr_handle,
)
print("Backend PID:", backend_process.pid)
print("stdout:", backend_stdout_path)
print("stderr:", backend_stderr_path)


## 4. Live Status + Final Validation

This waits for the backend and viewer, then prints the final validation summary. Sender-send-to-visualization timing is not recorded; the timing table uses receiver capture completion and local decoded CSV availability as the closest available proxies.


In [ ]:
rc = backend_process.wait()
for handle in (backend_stdout_handle, backend_stderr_handle):
    if not handle.closed:
        handle.close()

live_viewer["thread"].join(timeout=120)
if live_viewer["thread"].is_alive():
    live_viewer["stop_event"].set()
    live_viewer["thread"].join(timeout=10)

manifest_path = RUN_DIR / "reports/manifest.json"
summary_path = RUN_DIR / "reports/summary.json"
timing_path = RUN_DIR / "reports/live_visualization_timing.csv"
manifest = json.loads(manifest_path.read_text())
summary = json.loads(summary_path.read_text())
chunks = manifest.get("chunks", [])
validation = manifest.get("validation_result", {})
comparison = validation.get("comparison", {})
decoded_packets = sum(int((chunk.get("decode_export_statistics") or {}).get("packets_read") or 0) for chunk in chunks)
decoded_rows = sum(int((chunk.get("decode_export_statistics") or {}).get("rows_written") or 0) for chunk in chunks)
decoded_chunk_csvs = sorted((RUN_DIR / "artifacts/decoded_chunks").glob("chunk_*.csv"))

final_summary = {
    "backend_exit_code": rc,
    "run_id": RUN_ID,
    "run_dir": str(RUN_DIR),
    "chunk_csv_count": len(decoded_chunk_csvs),
    "decoded_packets": decoded_packets,
    "decoded_rows": decoded_rows,
    "raw_packet_drops": (manifest.get("queue_metrics") or {}).get("raw_packet_drops"),
    "validation_pass": validation.get("pass"),
    "missing_rows": comparison.get("missing_row_count"),
    "extra_rows": comparison.get("extra_row_count"),
    "viewer_updates": live_viewer["state"].get("updates"),
    "viewer_last_progress": (live_viewer["state"].get("last_result") or {}).get("progress_label"),
    "timing_csv": str(timing_path),
}
print(json.dumps(final_summary, indent=2, sort_keys=True))
if rc != 0:
    raise RuntimeError(f"backend failed; see {backend_stderr_path}")
if not timing_path.is_file():
    raise FileNotFoundError(timing_path)


## 5. Per-Chunk Timing / Lag Table

This table shows the receiver-side live visualization path in order: capture complete -> decode start -> decode complete -> local CSV available -> visualized. Breakdown columns are computed from the adjacent timestamps when the timing CSV does not already provide them. Sender-send-to-visualization timing is not recorded here, and unavailable timestamps are left blank rather than inferred.


In [ ]:
timing = pd.read_csv(timing_path)

def _ensure_duration_column(frame, column, start_col, end_col):
    if column in frame.columns:
        return
    if start_col not in frame.columns or end_col not in frame.columns:
        return
    start = pd.to_datetime(frame[start_col], utc=True, errors="coerce")
    end = pd.to_datetime(frame[end_col], utc=True, errors="coerce")
    frame[column] = (end - start).dt.total_seconds()

_ensure_duration_column(timing, "capture_to_decode_s", "receiver_capture_completed_at", "decode_started_at")
_ensure_duration_column(timing, "decode_duration_s", "decode_started_at", "decode_completed_at")
_ensure_duration_column(timing, "receiver_to_local_copy_s", "decode_completed_at", "local_chunk_csv_available_at")
_ensure_duration_column(timing, "visualization_lag_s", "local_chunk_csv_available_at", "visualized_at")
_ensure_duration_column(timing, "capture_to_visualization_lag_s", "receiver_capture_completed_at", "visualized_at")

slide_cols = [
    "chunk",
    "packets",
    "receiver_capture_completed_at",
    "decode_started_at",
    "capture_to_decode_s",
    "decode_completed_at",
    "decode_duration_s",
    "local_chunk_csv_available_at",
    "receiver_to_local_copy_s",
    "visualized_at",
    "visualization_lag_s",
    "capture_to_visualization_lag_s",
]
missing_cols = [col for col in slide_cols if col not in timing.columns]
if missing_cols:
    raise KeyError(f"timing CSV missing expected columns: {missing_cols}")
display(timing[slide_cols])


## 6. Final Heatmap + Ridgeline

Final live-viewer outputs after the completed run. The heatmap uses the same cumulative chunk CSVs as the live updates and displays `ADC - pedestal` with fixed color limits `[-20, 50]`.


In [ ]:
from IPython.display import HTML, display
import base64
import html as html_module


def _png_data_uri(path):
    return "data:image/png;base64," + base64.b64encode(Path(path).read_bytes()).decode("ascii")


def display_live_image_pair(heatmap_path, waveform_path, progress_label=None):
    heatmap_path = Path(heatmap_path)
    waveform_path = Path(waveform_path)
    missing = [path for path in (heatmap_path, waveform_path) if not path.is_file()]
    label_suffix = f" — {html_module.escape(str(progress_label))}" if progress_label else ""
    if missing:
        for path in missing:
            print("missing", path)
        return
    html = f"""
    <div style="display:flex;gap:10px;align-items:flex-start;justify-content:center;width:100%;max-width:1100px;margin:0 auto;background:#080b0f;padding:8px;box-sizing:border-box;overflow:hidden;">
      <div style="flex:0 1 49%;max-width:49%;min-width:0;text-align:center;color:#f3f6f9;font-family:sans-serif;box-sizing:border-box;">
        <div style="font-weight:700;margin:0 0 6px 0;">Heatmap{label_suffix}</div>
        <img src="{_png_data_uri(heatmap_path)}" style="width:100%;height:auto;object-fit:contain;display:block;border:1px solid #263442;border-radius:4px;" />
      </div>
      <div style="flex:0 1 49%;max-width:49%;min-width:0;text-align:center;color:#f3f6f9;font-family:sans-serif;box-sizing:border-box;">
        <div style="font-weight:700;margin:0 0 6px 0;">Ridgeline{label_suffix}</div>
        <img src="{_png_data_uri(waveform_path)}" style="width:100%;height:auto;object-fit:contain;display:block;border:1px solid #263442;border-radius:4px;" />
      </div>
    </div>
    """
    display(HTML(html))


heatmap_path = RUN_DIR / "visualizations/live_chunks/live_heatmap.png"
waveform_path = RUN_DIR / "visualizations/live_chunks/live_stacked_waveform.png"
print(heatmap_path)
print(waveform_path)
display_live_image_pair(
    heatmap_path,
    waveform_path,
    progress_label=(live_viewer["state"].get("last_result") or {}).get("progress_label") if "live_viewer" in globals() else None,
)
